# Warp-MD CUDA deterministic Colab (water/salt)

This notebook runs a **deterministic** water+salt MD simulation on Colab GPU,
saves the trajectory to Google Drive for reuse, then compares **warp-md** vs
**MDAnalysis** on MSD accuracy and runtime.

**Tip:** Set `REPO_URL` to your fork. If private, add `GITHUB_TOKEN` in Colab Secrets.


In [26]:
!nvidia-smi
import os, sys, time, shutil, json, math


Fri Jan 30 05:00:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/warp-md'
os.makedirs(DRIVE_ROOT, exist_ok=True)


Mounted at /content/drive


In [20]:
import os
from getpass import getpass

# 1. This creates a secure input box. Paste your token there.
# It will not be visible and won't be saved in the notebook file.
token = getpass('Paste your GitHub Token: ')
os.environ['GITHUB_TOKEN'] = token

# 2. Now run your clone script
%cd /content
!rm -rf warp-md

REPO_URL = 'https://github.com/Haayhur/warp-md.git'
repo_host = REPO_URL.replace('https://', '')

# Use the token we just captured securely
if os.environ.get('GITHUB_TOKEN'):
    print("Token detected. Cloning...")
    # Using --quiet to prevent printing the URL with token in case of errors
    !git clone --depth 1 --quiet https://{os.environ['GITHUB_TOKEN']}@{repo_host} warp-md
else:
    !git clone --depth 1 {REPO_URL} warp-md

%cd warp-md

/content
Token detected. Cloning...
Token detected. Cloning...
/content/warp-md


In [21]:
!curl https://sh.rustup.rs -sSf | sh -s -- -y
os.environ['PATH'] = os.environ['HOME'] + '/.cargo/bin:' + os.environ['PATH']
!rustc --version


info: downloading installer
warn: It looks like you have an existing rustup settings file at:
warn: /root/.rustup/settings.toml
warn: Rustup will install the default toolchain as specified in the settings file,
warn: instead of the one inferred from the default host triple.
info: profile set to 'default'
info: default host triple is x86_64-unknown-linux-gnu
warn: Updating existing toolchain, profile choice will be ignored
info: syncing channel updates for 'stable-x86_64-unknown-linux-gnu'
info: default toolchain set to 'stable-x86_64-unknown-linux-gnu'

  stable-x86_64-unknown-linux-gnu unchanged - rustc 1.93.0 (254b59607 2026-01-19)


Rust is installed now. Great!

To get started you may need to restart your current shell.
This would reload your PATH environment variable to include
Cargo's bin directory ($HOME/.cargo/bin).

To configure your current shell, you need to source
the corresponding env file under $HOME/.cargo.

This is usually done by running one of the following (note the 

In [ ]:
!python3 -m venv /content/warp_env

: 

: 

In [30]:
!pip -q install maturin openmm mdanalysis numpy pandas


In [31]:
# Build warp-md Python bindings with CUDA
!maturin develop --release --features cuda


💥 maturin failed
  Caused by: Couldn't find a virtualenv or conda environment, but you need one to use this command. For maturin to find your virtualenv you need to either set VIRTUAL_ENV (through activate), set CONDA_PREFIX (through conda activate) or have a virtualenv called .venv in the current or any parent folder. See https://virtualenv.pypa.io/en/latest/index.html on how to use virtualenv or use `maturin build` and `pip install <path/to/wheel>` instead.


In [ ]:
# Deterministic water/salt MD (OpenMM), export to Drive for reuse
import io
import openmm.app as app
from openmm import unit, Platform, LangevinIntegrator
from openmm.app import PDBFile, Modeller, ForceField, Simulation, DCDReporter, StateDataReporter

local_dir = '/content/warp-md/data'
drive_dir = f'{DRIVE_ROOT}/trajectories/water_salt'
os.makedirs(local_dir, exist_ok=True)
os.makedirs(drive_dir, exist_ok=True)
pdb_path = f'{local_dir}/water_salt.pdb'
dcd_path = f'{local_dir}/water_salt.dcd'
drive_pdb = f'{drive_dir}/water_salt.pdb'
drive_dcd = f'{drive_dir}/water_salt.dcd'
if os.path.exists(drive_pdb) and os.path.exists(drive_dcd):
    shutil.copy(drive_pdb, pdb_path)
    shutil.copy(drive_dcd, dcd_path)
    print('Using cached trajectory from Drive')
else:
    # Small alanine dipeptide template for solvation
    pdb_str = '''
ATOM      1  N   ACE A   1       0.000   0.000   0.000  1.00  0.00           N
ATOM      2  CH3 ACE A   1       0.000   0.000   1.500  1.00  0.00           C
ATOM      3  C   ACE A   1       1.200   0.000  -0.600  1.00  0.00           C
ATOM      4  O   ACE A   1       2.400   0.000  -0.600  1.00  0.00           O
ATOM      5  N   NME A   2       0.800   0.000  -1.800  1.00  0.00           N
ATOM      6  CH3 NME A   2       1.600   0.000  -3.000  1.00  0.00           C
TER
END
'''
    pdb = PDBFile(io.StringIO(pdb_str))
    forcefield = ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')
    modeller = Modeller(pdb.topology, pdb.positions)
    modeller.addSolvent(
        forcefield,
        model='tip3p',
        padding=1.0 * unit.nanometer,
        ionicStrength=0.15 * unit.molar,
        neutralize=True,
    )
    system = forcefield.createSystem(
        modeller.topology,
        nonbondedMethod=app.PME,
        nonbondedCutoff=1.0 * unit.nanometer,
        constraints=app.HBonds,
    )
    integrator = LangevinIntegrator(
        300 * unit.kelvin,
        1.0 / unit.picosecond,
        2.0 * unit.femtoseconds,
    )
    integrator.setRandomNumberSeed(1234)
    platform = Platform.getPlatformByName('CUDA')
    properties = {'CudaPrecision': 'mixed'}
    try:
        properties['DeterministicForces'] = 'true'
    except Exception:
        pass
    sim = Simulation(modeller.topology, system, integrator, platform, properties)
    sim.context.setPositions(modeller.positions)
    sim.context.setVelocitiesToTemperature(300 * unit.kelvin, 1234)
    sim.minimizeEnergy()
    with open(pdb_path, 'w') as handle:
        PDBFile.writeFile(modeller.topology, modeller.positions, handle)
    n_steps = 2000
    report_interval = 10
    sim.reporters.append(DCDReporter(dcd_path, report_interval))
    sim.reporters.append(StateDataReporter(sys.stdout, 200, step=True, temperature=True, potentialEnergy=True))
    sim.step(n_steps)
    shutil.copy(pdb_path, drive_pdb)
    shutil.copy(dcd_path, drive_dcd)
    print('Saved trajectory to Drive')


In [ ]:
# warp-md MSD (GPU)
import numpy as np
import warp_md as wmd
from warp_md import group_types_from_selections

system = wmd.System.from_pdb(pdb_path)
atoms = system.atom_table()
resnames = sorted(set(atoms['resname']))
print('resnames:', resnames)

def pick_name(candidates, default):
    for name in candidates:
        if name in resnames:
            return name
    return default

cation = pick_name(['NA', 'Na', 'Na+', 'SOD'], resnames[-2] if len(resnames) >= 2 else resnames[0])
anion = pick_name(['CL', 'Cl', 'Cl-', 'CLA'], resnames[-1])
sel_cat = f'resname {cation}'
sel_ani = f'resname {anion}'
sel_str = f'{sel_cat} or {sel_ani}'
selection = system.select(sel_str)
group_types = group_types_from_selections(system, selection, 'resid', [sel_cat, sel_ani])

traj = wmd.Trajectory.open_dcd(dcd_path, system)
msd = wmd.MsdPlan(selection, group_by='resid', group_types=group_types, lag_mode='fft')
t0 = time.perf_counter()
t_w, data_w = msd.run(traj, system, device='cuda')
t1 = time.perf_counter()
t_w = np.asarray(t_w)
data_w = np.asarray(data_w)
n_types = max(group_types) + 1
block = n_types + 1
msd_total_col = block * 3 + n_types
msd_w = data_w[:, msd_total_col]
warp_md_time = t1 - t0
print('warp-md MSD shape:', msd_w.shape, 'time:', warp_md_time)


In [ ]:
# MDAnalysis MSD
import MDAnalysis as mda
from MDAnalysis.analysis.msd import EinsteinMSD

u = mda.Universe(pdb_path, dcd_path)
t0 = time.perf_counter()
msd_mda = EinsteinMSD(u, select=sel_str, msd_type='xyz', fft=True)
msd_mda.run()
t1 = time.perf_counter()
md_time = t1 - t0

if hasattr(msd_mda.results, 'timeseries'):
    md_vals = np.asarray(msd_mda.results.timeseries)
elif hasattr(msd_mda.results, 'msds'):
    md_vals = np.asarray(msd_mda.results.msds)
else:
    md_vals = np.asarray(msd_mda.timeseries)

# md_vals may include t=0; align lengths
md_vals = md_vals.squeeze()
print('MDAnalysis MSD shape:', md_vals.shape, 'time:', md_time)


In [ ]:
# Accuracy + speed summary
md_cmp = md_vals
if len(md_vals) == len(msd_w) + 1 and abs(md_vals[0]) < 1e-6:
    md_cmp = md_vals[1:]

min_len = min(len(msd_w), len(md_cmp))
msd_w_cmp = msd_w[:min_len]
md_cmp = md_cmp[:min_len]
abs_err = np.max(np.abs(msd_w_cmp - md_cmp))
rel_err = np.max(np.abs(msd_w_cmp - md_cmp) / np.maximum(1e-8, np.abs(md_cmp)))
print(f'MSD compare over {min_len} points')
print('max abs err:', abs_err)
print('max rel err:', rel_err)
print('warp-md time (s):', warp_md_time)
print('MDAnalysis time (s):', md_time)
